In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tsgm
import tensorflow as tf
from tensorflow import keras
import time

In [2]:
#Save Kyoto_Gases data (2020-2100), consider the case of C1-C8 for simplicity.
Kyoto_Gases = pd.read_csv('Kyoto Gases.csv')
Kyoto_Gases = Kyoto_Gases[Kyoto_Gases['Category'].isin(['C1','C2','C3','C4','C5','C6','C7','C8'])]
mapping = {'C1':0,'C2':0,'C3':0,'C4':0,'C5':1,'C6':1,'C7':2,'C8':2}#Aggregate categories into 3 categories, with 0-2 corresponding to C1234-C78, respectively
Kyoto_Gases['Category'].replace(mapping,inplace=True)
Kyoto_Gases.reset_index(drop=True,inplace=True)
Kyoto_Gases.drop(columns=['Category_name'],inplace = True)

In [3]:
#Load a dataset of individual variables
CarbonSequestration = pd.read_csv('Carbon_Sequestration_CCS_imputed.csv')
FinalEnergy_Liquid = pd.read_csv('Final Energy_Liquids.csv')
PrimaryEnergy_Gas = pd.read_csv('Primary Energy_Gas.csv')
PrimaryEnergy_Oil = pd.read_csv('Primary Energy_Oil.csv')
PrimaryEnergy_Coal = pd.read_csv('PrimaryEnergy_Coal.csv')
SecondaryEle_Nuclear = pd.read_csv('SecondaryEnergy_Electricity_Nuclear.csv')
SecondaryEle_Hydro = pd.read_csv('SecondaryEnergy_Electricity_Hydro.csv')
SecondaryEle = pd.read_csv('SecondaryEnergy_Electricity.csv')
SecondaryEle_Oil = pd.read_csv('SecondaryEnergy_Electricity_Oil.csv')
SecondaryEle_Coal = pd.read_csv('SecondaryEnergy_Electricity_Coal.csv')
SecondaryEle_Gas = pd.read_csv('SecondaryEnergy_Electricity_Gas.csv')
SecondaryEle_Wind = pd.read_csv('SecondaryEnergy_Electricity_Wind.csv')
SecondaryEle_Solar = pd.read_csv('SecondaryEnergy_Electricity_Solar.csv')
SecondaryEle_Biomass = pd.read_csv('SecondaryEnergy_Electricity_Biomass.csv')
SecondaryEle_Geothermal = pd.read_csv('SecondaryEnergy_Electricity_Geothermal.csv')

In [4]:
#Get the intersection of the models and scenarios contained in each variable
Model_Scenario = Kyoto_Gases[['Model','Scenario']]
Variables = [CarbonSequestration,FinalEnergy_Liquid,PrimaryEnergy_Coal,PrimaryEnergy_Gas,PrimaryEnergy_Oil,SecondaryEle_Nuclear,
            SecondaryEle_Oil,SecondaryEle_Solar,SecondaryEle_Wind,SecondaryEle_Hydro,SecondaryEle_Geothermal,SecondaryEle_Gas,
            SecondaryEle_Coal,SecondaryEle_Biomass,SecondaryEle]
for variable in Variables:
    Model_Scenario = pd.merge(Model_Scenario,variable[['Model','Scenario']],on=['Model','Scenario'],how='inner')

In [5]:
for i in range(len(Variables)):
    Variables[i] = pd.merge(Model_Scenario,Variables[i],on=['Model','Scenario'],how='inner')
for i in range(len(Variables)):
    Variables[i].drop(columns=['Category_name'],inplace = True)

In [6]:
Kyoto_Gases = pd.merge(Kyoto_Gases,Model_Scenario,on = ['Model','Scenario'],how = 'inner')

In [7]:
Variables.append(Kyoto_Gases)
data_num = Variables[0].shape[0]

In [8]:
#ntf:the num of feature
nft = len(Variables)

X = np.zeros((data_num,9,nft))
for i in range(nft):
    Variables[i] = Variables[i].iloc[:,3:-1].values
    
for i in range(data_num):
    for j in range(9):
        for k in range(nft):
            X[i][j][k] = (Variables[k])[i,j]

In [9]:
Y = Kyoto_Gases['Category'].values

In [10]:
#Separate datasets by category. But the training process does not distinguish between categories like VAE
C1234_DataSet = X[Y == 0]
C56_DataSet = X[Y == 1]
C78_DataSet = X[Y == 2]
#the num of C1234,C56,C78
C1234_DataSet.shape[0],C56_DataSet.shape[0],C78_DataSet.shape[0]

(554, 237, 110)

In [11]:
#Define the model structure for C1234
architecture1 = tsgm.models.zoo["vae_conv5"](9, nft, 10)#Latent Dim = 10,number of features = nft
encoder1, decoder1 = architecture1.encoder, architecture1.decoder
encoder1.load_weights('Policy-encoder1_weights-Se.h5')
decoder1.load_weights('Policy-decoder1_weights-Se.h5')

In [12]:
#Standardize the data and compress it to (0, 1).
scaler_C1234 = tsgm.utils.TSFeatureWiseScaler((0,1))        
scaled_C1234_data = scaler_C1234.fit_transform(C1234_DataSet)

In [13]:
#Define the model structure for C56
architecture2 = tsgm.models.zoo["vae_conv5"](9, nft, 10)
encoder2, decoder2 = architecture2.encoder, architecture2.decoder
encoder2.load_weights('Policy-encoder2_weights-Se.h5')
decoder2.load_weights('Policy-decoder2_weights-Se.h5')

scaler_C56 = tsgm.utils.TSFeatureWiseScaler((0,1))    
scaled_C56_data = scaler_C56.fit_transform(C56_DataSet)

In [14]:
#Define the model structure for C78
architecture3 = tsgm.models.zoo["vae_conv5"](9, nft, 10)
encoder3, decoder3 = architecture3.encoder, architecture3.decoder
encoder3.load_weights('Policy-encoder3_weights-Se.h5')
decoder3.load_weights('Policy-decoder3_weights-Se.h5')

scaler_C78 = tsgm.utils.TSFeatureWiseScaler((0,1))    
scaled_C78_data = scaler_C78.fit_transform(C78_DataSet)

In [15]:
#Set Global Random Seed
global_seed = 3
tf.random.set_seed(global_seed)
np.random.seed(global_seed)

In [16]:
#Generate data using generative models (10000 for each class)
z1 = tf.random.normal((10000, 10))
z2 = tf.random.normal((10000, 10))
z3 = tf.random.normal((10000, 10))

Gen_C1234 = decoder1(z1)
Gen_C56 = decoder2(z2)
Gen_C78 = decoder3(z3)
Gen_C1234 = scaler_C1234.inverse_transform(Gen_C1234).numpy()
Gen_C56 = scaler_C56.inverse_transform(Gen_C56).numpy()
Gen_C78 = scaler_C78.inverse_transform(Gen_C78).numpy()




In [17]:
#Compute the cumulative value of the five input variables as a feature of the real dataset
features_names = ['CarbonSequestration','FinalEnergy_Liquid','PrimaryEnergy_Coal','PrimaryEnergy_Gas','PrimaryEnergy_Oil','SecondaryEle_Nuclear',
                 'SecondaryEle_Oil','SecondaryEle_Solar','SecondaryEle_Wind','SecondaryEle_Hydro','SecondaryEle_Geothermal','SecondaryEle_Gas',
                 'SecondaryEle_Coal','SecondaryEle_Biomass','SecondaryEle','Kyoto_Gases']


In [22]:
import os

def convert_to_wide_format_and_save(data_array, feature_names, output_dir='Vae-SecondaryEle-data', time_steps=None):
    """
    Converts a 3D array of shape [Index × Time Steps × Features] into a wide-format 
    pandas DataFrame and saves it as a CSV file.
    
    Parameters:
    data_array: numpy array of shape [n_indices, n_time_steps, n_features]
    feature_names: List of feature variable names.
    output_dir: Output directory, defaults to 'Vae-SecondaryEle-data'.
    time_steps: List of time steps, defaults to years 2020 to 2100 with a 10-year interval.
    
    Returns:
    df: pandas DataFrame with identifiers (Scenario, Variable) and time steps as columns.
    csv_path: The file path where the CSV was saved.
    """
    # Get array dimensions
    n_indices, n_time_steps, n_features = data_array.shape
    
    # Use default time steps (2020-2100, interval of 10) if none provided
    if time_steps is None:
        time_steps = list(range(2020, 2101, 10))
    
    # Validate that feature name list length matches feature count
    if len(feature_names) != n_features:
        raise ValueError(f"Feature names list length ({len(feature_names)}) does not match feature count ({n_features})")
    
    # Validate that time steps list length matches time steps count
    if len(time_steps) != n_time_steps:
        raise ValueError(f"Time steps list length ({len(time_steps)}) does not match time steps count ({n_time_steps})")
    
    # Create an empty list to store records
    data_records = []
    
    # Restructure data
    for i in range(n_indices):
        for f_idx, feature in enumerate(feature_names):
            record = {
                'Scenario': 'Scenario:' + str(i + 1),
                'Variable': feature
            }
            # Add data for each time step as a separate column
            for t_idx, year in enumerate(time_steps):
                record[str(year)] = data_array[i, t_idx, f_idx]
            
            data_records.append(record)
    
    # Create DataFrame and sort by Scenario
    df = pd.DataFrame(data_records)
    df = df.sort_values(by='Scenario')
    
    # Create output directory if it does not exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Generate CSV file path
    csv_path = os.path.join(output_dir, "C1234.csv")
    
    # Save DataFrame to CSV with UTF-8-SIG encoding
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    
    print(f"Data successfully saved to: {csv_path}")
    
    return df

In [23]:
convert_to_wide_format_and_save(data_array=Gen_C1234,feature_names=features_names)

数据已保存到: Vae-SecondaryEle-data\C1234.csv


,Scenario,Variable,2020,2030,2040,2050,2060,2070,2080,2090,2100
0,Scenario:1,CarbonSequestration,15.333172,553.622864,2423.235352,5072.067871,8550.335938,10815.941406,13539.057617,14528.168945,14150.821289
14,Scenario:1,SecondaryEle,95.691696,114.470963,155.957993,221.381653,278.494263,362.682190,416.164612,459.873840,509.702515
13,Scenario:1,SecondaryEle_Biomass,1.273692,1.675407,1.051481,1.557668,2.387882,3.235656,3.698764,3.998686,4.588403
12,Scenario:1,SecondaryEle_Coal,34.144657,5.016679,0.749880,0.229185,0.323324,0.273548,0.534228,0.525101,0.643592
11,Scenario:1,SecondaryEle_Gas,23.439360,33.517929,33.083702,28.160515,33.207901,28.540894,30.924192,28.370159,26.336607
...,...,...,...,...,...,...,...,...,...,...,...
159972,Scenario:9999,PrimaryEnergy_Oil,223.154144,231.881332,183.377808,111.415398,55.184383,23.887188,12.856899,4.479516,3.999664
159971,Scenario:9999,PrimaryEnergy_Gas,122.031807,172.568512,215.243393,233.145752,225.178253,190.047913,141.636063,87.717094,56.437325
159970,Scenario:9999,PrimaryEnergy_Coal,133.520828,50.923458,43.552269,61.824505,83.590469,96.346146,80.564323,45.746574,19.246719
159983,Scenario:9999,Kyoto_Gases,54971.359375,43179.476562,32465.548828,19428.810547,7870.101562,945.630859,-3973.091797,-8102.720703,-9517.605469


In [30]:
def extract_and_save_features(tensor, time_steps, feature_names, output_dir='Vae-SecondaryEle-data'):
    """
    Extracts time-series data for each feature from a 3D tensor and saves them as individual CSV files.
    
    Parameters:
    tensor: 3D numpy array of shape [Index x Time Step x Feature].
    time_steps: List of time steps, e.g., [2020, 2030, ..., 2100].
    feature_names: List of feature names, length must match the third dimension of the tensor.
    output_dir: Directory where the CSV files will be saved.
    """
    # Create the output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Get dimension sizes
    n_indices, n_timesteps, n_features = tensor.shape
    
    # Validate consistency of input dimensions
    assert len(time_steps) == n_timesteps, "Number of time steps does not match the second dimension of the tensor."
    assert len(feature_names) == n_features, "Number of feature names does not match the third dimension of the tensor."
    
    # Create a CSV file for each feature
    for f_idx, feature_name in enumerate(feature_names):
        # Extract data for a specific feature across all indices and time steps
        feature_data = tensor[:, :, f_idx]  # Shape: [Index x Time Step]
        
        # Create a DataFrame where rows are indices and columns are time steps
        df = pd.DataFrame(feature_data, columns=time_steps)
        
        # Save to CSV
        filename = 'C78_' + f"{feature_name.replace(' ', '_')}.csv"
        file_path = os.path.join(output_dir, filename)
        df.to_csv(file_path, index=False, encoding='utf-8-sig')
        
        print(f"Saved feature '{feature_name}' to file: {file_path}")
    
    print(f"All {n_features} features have been successfully exported to the '{output_dir}' directory.")

In [31]:
years = [str(i) for i in range(2020, 2101, 10)]
extract_and_save_features(tensor=Gen_C78,feature_names=features_names,time_steps=years)

已保存特征 'CarbonSequestration' 到文件: Vae-SecondaryEle-data\C78_CarbonSequestration.csv
已保存特征 'FinalEnergy_Liquid' 到文件: Vae-SecondaryEle-data\C78_FinalEnergy_Liquid.csv
已保存特征 'PrimaryEnergy_Coal' 到文件: Vae-SecondaryEle-data\C78_PrimaryEnergy_Coal.csv
已保存特征 'PrimaryEnergy_Gas' 到文件: Vae-SecondaryEle-data\C78_PrimaryEnergy_Gas.csv
已保存特征 'PrimaryEnergy_Oil' 到文件: Vae-SecondaryEle-data\C78_PrimaryEnergy_Oil.csv
已保存特征 'SecondaryEle_Nuclear' 到文件: Vae-SecondaryEle-data\C78_SecondaryEle_Nuclear.csv
已保存特征 'SecondaryEle_Oil' 到文件: Vae-SecondaryEle-data\C78_SecondaryEle_Oil.csv
已保存特征 'SecondaryEle_Solar' 到文件: Vae-SecondaryEle-data\C78_SecondaryEle_Solar.csv
已保存特征 'SecondaryEle_Wind' 到文件: Vae-SecondaryEle-data\C78_SecondaryEle_Wind.csv
已保存特征 'SecondaryEle_Hydro' 到文件: Vae-SecondaryEle-data\C78_SecondaryEle_Hydro.csv
已保存特征 'SecondaryEle_Geothermal' 到文件: Vae-SecondaryEle-data\C78_SecondaryEle_Geothermal.csv
已保存特征 'SecondaryEle_Gas' 到文件: Vae-SecondaryEle-data\C78_SecondaryEle_Gas.csv
已保存特征 'SecondaryEle_Coal' 